# Sentinel-Time-Viewer
Comparador temporal de imagens Sentinel-2 Surface Reflectance (Harmonized) com controle deslizante e sobreposicao vetorial persistente.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NGEO-IDEFLOR-Bio/Sentinel-Time-Viewer/blob/main/GEE_swipe.ipynb)

Nucleo de Geotecnologias (NGEO) | Instituto de Desenvolvimento Florestal e da Biodiversidade do Estado do Para (IDEFLOR-Bio)

## Instrucoes de Execucao
1. Celula 1: Instalacao das dependencias no ambiente Colab.
2. Celula 2: Configuracao do GCP Project ID e autenticacao no Google Earth Engine.
3. Celula 3: Ajuste de parametros de filtro de nuvens, meses e selecao de basemap.
4. Celula 4: Carregamento do poligono vetorial delimitador (Shapefile .zip ou demonstrativo).
5. Celula 5: Verificacao automatica de anos disponiveis e renderizacao do visualizador interativo.
6. Celula 6: Exportacao da comparacao entre dois anos selecionados em formato GIF animado.

In [ ]:
# @title 1. Instalacao de Dependencias { display-mode: "form" }
# @markdown Execucao do comando pip para instalacao das bibliotecas necessarias no ambiente Colab.

!pip install -q --upgrade earthengine-api geemap ipywidgets pycrs pillow requests
print("[OK] Dependencias instaladas com sucesso.")


In [ ]:
# @title 2. Autenticacao e Configuracao do Google Cloud Project { display-mode: "form" }
# @markdown Informe o identificador do projeto Google Cloud registrado para uso da API Earth Engine:

GCP_PROJECT_ID = "ee-samuelsantosambientalcourse"  # @param {type:"string"}

import ee

project_clean = GCP_PROJECT_ID.strip()
if not project_clean or project_clean == "seu-projeto-gcp":
    raise ValueError("Informe um GCP Project ID valido no campo acima.")

try:
    ee.Initialize(project=project_clean)
    print(f"[OK] Google Earth Engine inicializado no projeto: '{project_clean}'.")
except Exception as init_err:
    print(f"[INFO] Solicitando autenticacao para o projeto '{project_clean}'...")
    try:
        ee.Authenticate()
        ee.Initialize(project=project_clean)
        print(f"[OK] Autenticacao realizada e GEE inicializado no projeto: '{project_clean}'.")
    except Exception as auth_err:
        print(f"[ERRO] Falha ao autenticar no GEE com o projeto '{project_clean}': {auth_err}")
        raise auth_err


In [ ]:
# @title 3. Parametros de Filtragem e Mapa Base { display-mode: "form" }
# @markdown Definicao de limiares operacionais e configuracao cartografica:

# @markdown Percentual maximo tolerado de cobertura de nuvens por cena:
MAX_CLOUD_PERCENTAGE = 20  # @param {type:"slider", min:5, max:100, step:5}

# @markdown Mapa base para exibicao de contexto:
BASEMAP = "HYBRID"  # @param ["HYBRID", "SATELLITE", "ROADMAP", "Esri.WorldImagery", "Esri.WorldTopoMap", "OpenStreetMap"]

# @markdown Intervalo mensal para agregacao anual:
START_MONTH = 1  # @param {type:"slider", min:1, max:12, step:1}
END_MONTH = 12  # @param {type:"slider", min:1, max:12, step:1}

# @markdown Estilo da linha de contorno da area de interesse:
OUTLINE_COLOR = "#00FF00"  # @param {type:"string"}
OUTLINE_WIDTH = 2  # @param {type:"slider", min:1, max:5, step:1}

print("[CONFIG] Parametros definidos:")
print(f"  - Limite de nuvens: < {MAX_CLOUD_PERCENTAGE}%")
print(f"  - Meses de agregacao: {START_MONTH} a {END_MONTH}")
print(f"  - Mapa base: {BASEMAP}")
print(f"  - Contorno do poligono: cor {OUTLINE_COLOR}, espessura {OUTLINE_WIDTH}px")


In [ ]:
# @title 4. Definicao da Area de Interesse { display-mode: "form" }
# @markdown Selecione o metodo de insercao da geometria:

FONTE_AOI = "Upload de Shapefile (.zip)"  # @param ["Upload de Shapefile (.zip)", "Usar Poligono Demonstrativo (PE Utinga - Belem/PA)"]
FORCAR_NOVO_UPLOAD = False  # @param {type:"boolean"}

import os
import zipfile
import ee
import geemap
from google.colab import files

def carregar_geometria(fonte, forcar_novo):
    if fonte == "Usar Poligono Demonstrativo (PE Utinga - Belem/PA)":
        print("[INFO] Utilizando poligono delimitador de teste: Parque Estadual do Utinga (Belem/PA).")
        coords = [
            [-48.455, -1.435],
            [-48.410, -1.435],
            [-48.410, -1.405],
            [-48.455, -1.405],
            [-48.455, -1.435]
        ]
        return ee.Geometry.Polygon([coords])
    
    shp_dir = "shapefile"
    if not os.path.exists(shp_dir):
        os.makedirs(shp_dir)
        
    shp_existentes = [
        os.path.join(root, f)
        for root, _, files_list in os.walk(shp_dir)
        for f in files_list
        if f.lower().endswith(".shp") and not f.startswith(".") and "__MACOSX" not in root
    ]
    
    if shp_existentes and not forcar_novo:
        print(f"[INFO] Reutilizando Shapefile da sessao: {os.path.basename(shp_existentes[0])}")
        return geemap.shp_to_ee(shp_existentes[0]).geometry()
        
    for root, dirs, files_list in os.walk(shp_dir, topdown=False):
        for f in files_list:
            try: os.remove(os.path.join(root, f))
            except Exception: pass
        for d in dirs:
            try: os.rmdir(os.path.join(root, d))
            except Exception: pass
            
    print("[INFO] Selecione o arquivo .zip com os componentes do Shapefile (.shp, .shx, .dbf, .prj):")
    uploaded = files.upload()
    if not uploaded:
        print("[AVISO] Nenhum arquivo enviado.")
        return None
        
    zip_processado = False
    for fn in uploaded:
        if fn.lower().endswith(".zip"):
            with zipfile.ZipFile(fn, "r") as z:
                z.extractall(shp_dir)
            print(f"[OK] Arquivo '{fn}' extraido.")
            zip_processado = True
            break
            
    if not zip_processado:
        print("[ERRO] Nenhum arquivo .zip detectado.")
        return None
        
    shp_encontrados = [
        os.path.join(root, f)
        for root, _, files_list in os.walk(shp_dir)
        for f in files_list
        if f.lower().endswith(".shp") and not f.startswith(".") and "__MACOSX" not in root
    ]
    
    if not shp_encontrados:
        print("[ERRO] Nenhum arquivo .shp localizado dentro do pacote .zip.")
        return None
        
    shp_path = shp_encontrados[0]
    print(f"[OK] Shapefile carregado: {os.path.basename(shp_path)}")
    return geemap.shp_to_ee(shp_path).geometry()

region = carregar_geometria(FONTE_AOI, FORCAR_NOVO_UPLOAD)
if region:
    print("[OK] Geometria da area de interesse pronta para processamento.")


In [ ]:
# @title 5. Verificacao de Anos e Visualizador Temporal (Swipe) { display-mode: "form" }
# @markdown Consulta no catalogo do Earth Engine apenas os anos com dados validos e inicializa a interface:

import datetime
from IPython.display import display

if "region" not in globals() or region is None:
    raise ValueError("Area de interesse nao definida. Execute a Celula 4 primeiro.")

ano_atual = datetime.datetime.now().year
ano_inicial = 2017

print("[INFO] Consultando catalogo COPERNICUS/S2_SR_HARMONIZED...")
print(f"       Criterios: Nuvens < {MAX_CLOUD_PERCENTAGE}% | Meses: {START_MONTH} a {END_MONTH}")

def obter_anos_validos(geom, min_y, max_y, max_cloud, m_start, m_end):
    anos_lista = ee.List.sequence(min_y, max_y)
    def contar_cenas(y):
        y = ee.Number(y)
        d1 = ee.Date.fromYMD(y, m_start, 1)
        d2 = ee.Date.fromYMD(y, m_end, 1).advance(1, "month")
        coll = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                .filterBounds(geom)
                .filterDate(d1, d2)
                .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", max_cloud)))
        return ee.Feature(None, {"year": y, "count": coll.size()})
    
    fc = ee.FeatureCollection(anos_lista.map(contar_cenas))
    com_cenas = fc.filter(ee.Filter.gt("count", 0))
    feats = com_cenas.getInfo().get("features", [])
    return {int(f["properties"]["year"]): int(f["properties"]["count"]) for f in feats}

contagem_anos = obter_anos_validos(region, ano_inicial, ano_atual, MAX_CLOUD_PERCENTAGE, START_MONTH, END_MONTH)
anos_disponiveis = sorted(contagem_anos.keys())

if not anos_disponiveis:
    print(f"[AVISO] Nenhuma cena Sentinel-2 encontrada com cobertura de nuvens < {MAX_CLOUD_PERCENTAGE}% nesta area.")
    print("        Aumente o valor de MAX_CLOUD_PERCENTAGE na Celula 3 e execute novamente.")
elif len(anos_disponiveis) < 2:
    print(f"[AVISO] Apenas 1 ano com dados validos foi localizado: {anos_disponiveis[0]} ({contagem_anos[anos_disponiveis[0]]} cenas).")
    print("        Sao necessarios ao menos 2 anos distintos para a ferramenta de comparacao temporal.")
    print("        Amplie o limite de nuvens na Celula 3 para incluir outros periodos.")
else:
    print(f"[INFO] {len(anos_disponiveis)} anos com imagens validas identificados:")
    for a in anos_disponiveis:
        print(f"  - Ano {a}: {contagem_anos[a]:>3} cena(s)")
    
    print("[INFO] Gerando composicoes anuais medianas e contorno vetorial...")

    def rasterizar_borda(geom, cor_hex="#00FF00", largura=2):
        fc = ee.FeatureCollection([ee.Feature(geom)])
        cor_limpa = cor_hex.lstrip("#")
        borda = ee.Image().byte().paint(fc, 1, largura)
        return borda.visualize(palette=[cor_limpa], opacity=1)

    vis_rgb = {"min": 0, "max": 3000, "bands": ["B4", "B3", "B2"]}
    borda_vis = rasterizar_borda(region, OUTLINE_COLOR, OUTLINE_WIDTH)

    # Armazena colecoes em dicionario global para reutilizacao na exportacao
    mosaicos_anuais = {}
    imagens_mosaico = []
    for a in anos_disponiveis:
        d1 = ee.Date.fromYMD(a, START_MONTH, 1)
        d2 = ee.Date.fromYMD(a, END_MONTH, 1).advance(1, "month")
        coll = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                .filterBounds(region)
                .filterDate(d1, d2)
                .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", MAX_CLOUD_PERCENTAGE)))
        
        mediana = coll.median().select(["B4", "B3", "B2"]).visualize(**vis_rgb)
        combo = ee.ImageCollection([mediana, borda_vis]).mosaic().set("year", a)
        mosaicos_anuais[a] = combo
        imagens_mosaico.append(combo)

    try:
        Map = geemap.Map(basemap=BASEMAP)
    except Exception as err_map:
        print(f"[AVISO] Basemap '{BASEMAP}' indisponivel ({err_map}). Utilizando 'HYBRID'.")
        Map = geemap.Map(basemap="HYBRID")

    Map.centerObject(region, zoom=14)

    rotulos_anos = [str(a) for a in anos_disponiveis]
    Map.ts_inspector(
        left_ts=ee.ImageCollection(imagens_mosaico),
        right_ts=ee.ImageCollection(imagens_mosaico),
        left_names=rotulos_anos,
        right_names=rotulos_anos,
        left_vis={},
        right_vis={},
        width="90px"
    )

    display(Map)


In [ ]:
# @title 6. Exportar Animacao em GIF (Swipe) { display-mode: "form" }
# @markdown Parametros para geracao do arquivo GIF com efeito swipe e ticker informativo superior:

ANO_ESQUERDA = 2018  # @param {type:"integer"}
ANO_DIREITA = 2024  # @param {type:"integer"}
LARGURA_PIXELS = 768  # @param [512, 768, 1024]
DURACAO_QUADRO_MS = 80  # @param {type:"slider", min:40, max:200, step:10}
TEXTO_MARCA_DAGUA = "NGEO / IDEFLOR-Bio | Monitoramento Territorial"  # @param {type:"string"}
DOWNLOAD_AUTOMATICO = True  # @param {type:"boolean"}

import io
import requests
from PIL import Image, ImageDraw, ImageFont
import IPython.display

if "region" not in globals() or region is None:
    raise ValueError("Area de interesse nao encontrada. Execute as celulas anteriores primeiro.")

if "mosaicos_anuais" not in globals() or not mosaicos_anuais:
    raise ValueError("Mosaicos anuais nao disponiveis. Execute a Celula 5 primeiro.")

# Validacao dos anos informados
anos_disponiveis_lista = sorted(list(mosaicos_anuais.keys()))
ano_a = ANO_ESQUERDA if ANO_ESQUERDA in mosaicos_anuais else anos_disponiveis_lista[0]
ano_b = ANO_DIREITA if ANO_DIREITA in mosaicos_anuais else anos_disponiveis_lista[-1]

if ano_a == ano_b:
    if len(anos_disponiveis_lista) > 1:
        ano_a, ano_b = anos_disponiveis_lista[0], anos_disponiveis_lista[-1]
    else:
        raise ValueError("Sao necessarios ao menos dois anos distintos para compor o GIF.")

print(f"[INFO] Gerando animacao swipe: {ano_a} vs {ano_b} (Largura: {LARGURA_PIXELS}px)...")

def baixar_imagem_ano(ano, largura):
    img = mosaicos_anuais[ano]
    # Margem para evitar corte do poligono
    bounds = region.bounds(10)
    thumb_url = img.getThumbURL({
        "region": bounds,
        "dimensions": largura,
        "format": "png"
    })
    r = requests.get(thumb_url, timeout=90)
    if r.status_code != 200:
        raise RuntimeError(f"Falha ao transferir imagem do ano {ano} (status {r.status_code}).")
    return Image.open(io.BytesIO(r.content)).convert("RGB")

print(f"[INFO] Requisitando quadro do ano {ano_a}...")
img_a = baixar_imagem_ano(ano_a, LARGURA_PIXELS)
print(f"[INFO] Requisitando quadro do ano {ano_b}...")
img_b = baixar_imagem_ano(ano_b, LARGURA_PIXELS)

# Ajuste de dimensoes identicas
W, H = img_a.size
if img_b.size != (W, H):
    img_b = img_b.resize((W, H))

# Construcao da trajetoria do controle deslizante (36 passos no ciclo)
x_posicoes = []
for i in range(15):
    x_posicoes.append(int(W * (i / 14)))
for _ in range(3):
    x_posicoes.append(W)
for i in range(15):
    x_posicoes.append(int(W * (1 - i / 14)))
for _ in range(3):
    x_posicoes.append(0)

total_quadros = len(x_posicoes)
font = ImageFont.load_default()
texto_ticker = TEXTO_MARCA_DAGUA.strip()
try:
    largura_texto = int(font.getlength(texto_ticker))
except Exception:
    largura_texto = len(texto_ticker) * 7

deslocamento_total = W + largura_texto
quadros = []

for idx, x in enumerate(x_posicoes):
    quadro = Image.new("RGB", (W, H))
    
    # Lado esquerdo (ano_b) e lado direito (ano_a)
    if x > 0:
        quadro.paste(img_b.crop((0, 0, x, H)), (0, 0))
    if x < W:
        quadro.paste(img_a.crop((x, 0, W, H)), (x, 0))
        
    draw = ImageDraw.Draw(quadro)
    
    # Linha divisoria do swipe (abaixo da barra superior de 22px)
    if 0 < x < W:
        draw.line([(x, 22), (x, H)], fill=(255, 255, 255), width=2)
        
    # Ticker informativo continuo na borda superior
    offset_x = int(W - (idx / total_quadros) * deslocamento_total)
    draw.rectangle([(0, 0), (W, 22)], fill=(20, 20, 20))
    draw.text((offset_x, 4), texto_ticker, fill=(225, 225, 225), font=font)
    
    # Indicadores dos anos na base da imagem
    draw.rectangle([(10, H - 26), (65, H - 8)], fill=(10, 10, 10))
    draw.text((18, H - 23), str(ano_a), fill=(255, 255, 255), font=font)
    
    draw.rectangle([(W - 65, H - 26), (W - 10, H - 8)], fill=(10, 10, 10))
    draw.text((W - 57, H - 23), str(ano_b), fill=(255, 255, 255), font=font)
    
    quadros.append(quadro)

nome_arquivo_gif = f"swipe_{ano_a}_{ano_b}.gif"
quadros[0].save(
    nome_arquivo_gif,
    save_all=True,
    append_images=quadros[1:],
    duration=DURACAO_QUADRO_MS,
    loop=0
)

print(f"[OK] Animacao concluida: {nome_arquivo_gif} ({len(quadros)} quadros, {W}x{H}px).")
display(IPython.display.Image(nome_arquivo_gif))

if DOWNLOAD_AUTOMATICO:
    try:
        from google.colab import files
        files.download(nome_arquivo_gif)
        print(f"[INFO] Solicitando download do arquivo {nome_arquivo_gif}.")
    except Exception:
        print(f"[INFO] Arquivo salvo no diretorio da sessao: {nome_arquivo_gif}.")
